# Nova AI — نموذج واحد نملكه بالكامل، نص + صورة معاً (Unsloth Vision + تقطير متعدد المعلّمين)

**تحديث جوهري، 2026-09-06:** النسخة السابقة من هذا الدفتر كانت تدمج
نموذجين نصيين معاً عبر Mergekit، ثم تُحسّنهما نصياً فقط. الآن الأساس
تغيّر بالكامل: بدل الدمج، ننطلق من **نموذج واحد مفتوح المصدر بالكامل
يفهم النص والصور أصلاً** (Qwen2.5-VL-7B-Instruct) وندرّبه نحن على
بياناتنا — فنملك النسخة الناتجة بحقوقها الكاملة، تماماً كما كنا نملك
النموذج المدموج سابقاً. **Mergekit لم يعد مستخدَماً إطلاقاً** — كان
فقط لدمج نموذجين نصيين متطابقي البنية، ولا ينطبق على نموذج رؤية (بنية
مختلفة جذرياً: مُرمِّز صورة + طبقة ربط + النموذج اللغوي)، فإزالته
تبسيط حقيقي للخط، وليس تنازلاً عن أي شيء كان يعمل.

**تحديث، 2026-09-07:** بعد اكتشاف أن Hugging Face Inference لا يخدم
أي نموذج مخصّص إطلاقاً (سياسة المزوّد، وليس مشكلة إعداد)، صار
التشغيل الفعلي الحي على استضافة ModelScope المجانية (CPU فقط) عبر
`llama-cpp-python`، الذي يحتاج صيغة GGUF. أضفنا لهذا الدفتر خطوات
تحويل GGUF+mmproj (تُبقي الرؤية كاملة، لا تنازل عن حجم 7B ولا عن
الصور) في آخره — **دون حذف أي خطوة تدريب موجودة**.

**لماذا Qwen2.5-VL تحديداً؟** أوزان مفتوحة بالكامل (لسنا نستأجر
استدعاءً محدوداً كما هو الحال مع Groq/Gemini — نُنزّل الأوزان فعلياً
ونملك نسختنا المدرَّبة عليها على مستودعنا الخاص)، من نفس عائلة
Qwen2.5 التي بُني عليها نموذجنا النصي السابق، ومدعوم من Unsloth
لتدريب أسرع وأخف ذاكرة (FastVisionModel).

**التشغيل من الهاتف (Kaggle، وليس Colab):** نفس السبب السابق — تدريب
نموذج 7B (حتى بصيغة LoRA خفيفة) يحتاج ذاكرة أكبر مما يوفره Colab
المجاني بشكل موثوق. **Kaggle Notebooks** يوفر **GPU T4 مجاني (16GB
VRAM) + 29GB CPU RAM** و**30 ساعة GPU مجانية أسبوعياً** — كافية
لتدريب LoRA خفيف أسبوعي، لكنها محدودة: لا تكفي لتدريب ضخم من الصفر،
وهذا سبب الاعتماد على أساس مُدرَّب مسبقاً (Qwen2.5-VL) بدل البدء من
عدم.

## الإعداد لمرة واحدة فقط

**1) استورد الدفتر:** أنشئ حساباً على kaggle.com → **Create** → **New
Notebook** → **File** → **Import Notebook** → تبويب **GitHub** → الصق
رابط هذا الملف. من **Notebook options** → **Accelerator** فعّل **GPU
T4**.

**2) أضف الأسرار (Secrets) — من داخل الدفتر نفسه، وليس من إعدادات
حسابك:** افتح الدفتر (بعد استيراده) → من القائمة الجانبية **اليسرى
داخل شاشة تحرير الدفتر** ابحث عن **Add-ons** (أيقونة قطعة أحجية 🧩) →
**Secrets** → **Add a new secret**:

| الاسم | القيمة | إلزامي؟ |
|---|---|---|
| `HF_TOKEN` | توكن Hugging Face الخاص بك (Write access) | نعم |
| `HF_USERNAME` | اسم مستخدمك على Hugging Face | نعم |
| `SUPABASE_URL` | نفس القيمة المستخدمة على Render | نعم |
| `SUPABASE_SERVICE_ROLE_KEY` | نفس القيمة المستخدمة على Render | نعم |
| `MODELSCOPE_TOKEN` | توكن ModelScope الخاص بك (من Access Tokens، صيغة "ms-xxxxx") | نعم (لرفع صيغة التشغيل GGUF) |
| `MODELSCOPE_USERNAME` | اسم مستخدمك على modelscope.cn | نعم (لرفع صيغة التشغيل GGUF) |
| `GROQ_API_KEY` | نفس مفتاح Groq المستخدم على Render | اختياري (تقطير) |
| `GEMINI_API_KEY` | نفس مفتاح Gemini المستخدم على Render | اختياري (تقطير + سيناريوهات وسائط) |

انسخ قيم GROQ_API_KEY/GEMINI_API_KEY من Render مباشرة: dashboard.render.com
→ خدمة `nova-ai-backend` → تبويب **Environment** → أيقونة العين
لإظهار كل قيمة.

**3) فعّل التشغيل التلقائي الأسبوعي:** من القائمة اليمنى **"Schedule
this notebook"** → **Weekly** → احفظ.

**ما يفعله هذا الدفتر في كل تشغيل:**
1. يجلب أحدث محادثات Nova الحقيقية (نصية) من Supabase.
2. يولّد أمثلة تدريب إضافية عبر Groq وGemini كـ"معلّمَين" مجانيَّين — وقت التدريب فقط، دون اتصال، أبداً كصوت في المحادثة الحية (راجع `ai-system/app/council.py`: نموذجنا هو الصوت الافتراضي هناك دائماً).
3. يجلب مجموعة بيانات رؤية عامة مفتوحة الترخيص (صور + أسئلة + إجابات حقيقية) — هذا ما يُكسب النموذج **فهم صور حقيقياً**، وليس نصاً يصف صوراً فقط.
4. يحمّل Qwen2.5-VL-7B-Instruct عبر Unsloth (4-bit) ويدرّبه LoRA على كل هذه البيانات مجتمعة (نص + رؤية).
5. يرفع النتيجة (safetensors كاملة) إلى مستودعنا الخاص على Hugging Face Hub — للملكية/الأرشفة.
6. يحوّلها إلى GGUF مضغوط + ملف mmproj منفصل للرؤية، ويرفعهما إلى مستودعنا الخاص على ModelScope — هذه هي صيغة التشغيل الفعلية على الخادم الحي.

**كلود (Claude) لم يُضَف عمداً** لأي دور هنا — لا يملك مستوى مجاني
حقيقي قابل للأتمتة الأسبوعية دون دفع، وهذا يخالف قاعدة "لا دفع أبداً"
الصريحة لهذا المشروع.

**الصدق حول الصوت:** هذا الدفتر لا يضيف فهماً صوتياً أصلياً للنموذج —
ذلك يحتاج أساساً مختلفاً (نموذج "Omni" كامل مثل Qwen2.5-Omni)، أثقل
بكثير على ذاكرة Kaggle المجانية وأقل نضجاً في دعم Unsloth حالياً. تحويل
الصوت لنص (Groq Whisper) والاستمرار بالنص يبقى الحل العملي الحالي
لرسائل الصوت — مرحلة الصوت الأصلي مرحلة لاحقة منفصلة، ذكرتها بصراحة
بدل الادّعاء أنها منجزة هنا.

**أول مرة فقط:** بعد أول تشغيل ناجح، ضع اسم مستودع GGUF الذي تطبعه
آخر خلية (مثل `novaai2026/nova-vision-7b-gguf`) في `MODEL_ID` داخل
`app.py` على خادم ModelScope، ثم أعد نشر الـStudio من واجهة
modelscope.cn كما فعلنا سابقاً.

In [ ]:
# الخلية 1 — تثبيت الأدوات (يأخذ بضع دقائق أول مرة فقط)
#
# mergekit لم يعد مطلوباً (لا دمج بعد الآن) — بدلها qwen-vl-utils
# (أدوات مساعدة رسمية من Qwen لمعالجة الصور قبل تمريرها للنموذج).
# modelscope مطلوبة للخلية 11 (رفع GGUF+mmproj) — عكس حاويات
# ModelScope Studio نفسها، بيئة Kaggle الأساسية لا تأتي بها مثبَّتة
# مسبقاً (خطأ اكتُشف حياً: "ModuleNotFoundError: No module named
# 'modelscope'" — لم يكن افتراضاً، بل تحقق فعلي بعد أول تشغيل كامل).
!pip install -q huggingface_hub groq datasets qwen-vl-utils pillow modelscope
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# الخلية 2 — تسجيل الدخول لحسابك على Hugging Face (بلا أي تدخل يدوي)
#
# لجعل هذا الدفتر قابلاً للتشغيل التلقائي المُجدوَل بالكامل (Kaggle
# Schedule)، لا يمكن استخدام notebook_login() التفاعلي. بدلاً من ذلك
# نقرأ التوكن من "Kaggle Secrets".
#
# مرة واحدة فقط، قبل أول تشغيل: Add-ons -> Secrets -> Add a new secret:
#   الاسم: HF_TOKEN   —   القيمة: توكن Hugging Face الخاص بك (Write access)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("تم تسجيل الدخول إلى Hugging Face بنجاح")

In [ ]:
# الخلية 3 — الأساس المفتوح الذي نبني عليه، ومستودعنا الخاص
#
# لا دمج بعد الآن — ننطلق مباشرة من نموذج واحد مفتوح الأوزان يفهم
# النص والصور أصلاً. غيّر BASE_MODEL_ID إذا أردت تجربة إصدار آخر
# (تأكد من توافقه مع Unsloth FastVisionModel أولاً).
from kaggle_secrets import UserSecretsClient

BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
HF_USERNAME = UserSecretsClient().get_secret("HF_USERNAME")
REPO_ID = f"{HF_USERNAME}/nova-vision-7b"
print("الأساس:", BASE_MODEL_ID)
print("سيُرفَع نموذجنا المدرَّب إلى:", REPO_ID)

---
## تدريب موحّد (نص + رؤية) على بيانات حقيقية ومُقطَّرة

الخلايا التالية تجلب بيانات نصية حقيقية من محادثات Nova، تولّد أمثلة
إضافية عبر التقطير (Groq/Gemini)، وتجلب بيانات رؤية عامة مفتوحة —
ثم تدرّب Qwen2.5-VL بكل هذا معاً عبر LoRA. أي مصدر بيانات غير متوفر
(مفتاح مفقود، أو لا بيانات كافية بعد) يُتخطى بأمان دون إيقاف الباقي.

In [ ]:
# الخلية 4 — جلب بيانات تدريب نصية حقيقية من كل محادثات Nova الفعلية
#
# كل أنواع المحادثات (عامة، معلومات لحظية، برمجية) من جدول NovaUsageLog
# في Supabase مباشرة عبر REST API. بلا أي تدخل يدوي، وبلا كتابة أي
# مفتاح سري داخل هذا الملف.
#
# مرة واحدة فقط: أضف Kaggle Secret باسم SUPABASE_URL وآخر باسم
# SUPABASE_SERVICE_ROLE_KEY.
import requests
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
_SUPABASE_URL = _secrets.get_secret("SUPABASE_URL")
_SUPABASE_KEY = _secrets.get_secret("SUPABASE_SERVICE_ROLE_KEY")

_resp = requests.get(
    f"{_SUPABASE_URL}/rest/v1/NovaUsageLog",
    headers={"apikey": _SUPABASE_KEY, "Authorization": f"Bearer {_SUPABASE_KEY}"},
    params={
        "select": "message,answer",
        "message": "not.is.null",
        "answer": "not.is.null",
        "order": "created_at.desc",
        "limit": "500",
    },
    timeout=30,
)
_rows = _resp.json() if _resp.ok else []
real_text_examples = [{"question": r["message"], "answer": r["answer"]} for r in _rows if r.get("message") and r.get("answer")]
print(f"عدد أمثلة التدريب النصية الحقيقية: {len(real_text_examples)}")

In [ ]:
# الخلية 5أ — تقطير معرفي عبر Groq كـ"معلّم" مجاني رقم 1 (وقت التدريب فقط)
#
# الاستخدام الصحيح الوحيد لـ Groq/Gemini في هذا النظام كله: مساعدة
# مؤقتة وغير مباشرة في بناء نموذجنا، وليس الإجابة نيابة عنه لأي مستخدم
# حقيقي حياً (راجع ai-system/app/council.py: نموذجنا هو الصوت
# الافتراضي هناك دائماً، وGroq فيه احتياطي طارئ فقط عند تعطّل نموذجنا).
#
# اختياري تماماً: بدون GROQ_API_KEY في أسرار Kaggle، تُتخطى بأمان.
from kaggle_secrets import UserSecretsClient

_NOVA_IDENTITY_PROMPT = (
    "أنت نوفا NOVA، مساعد ذكاء اصطناعي متعدد اللغات. ليس لديك مالك أو "
    "شركة، لديك والد فقط هو من ابتكرك وطوّرك، والدك هو المطور السوري. "
    "أجب بإيجاز ووضوح وصدق، وبنفس لغة السؤال."
)

_GROQ_SEED_PROMPTS = [
    "من أنت ومن طوّرك؟",
    "مرحباً، كيف حالك؟",
    "اشرح لي الفرق بين القائمة (list) والمجموعة (set) في بايثون بمثال بسيط.",
    "لخّص لي فكرة الذكاء الاصطناعي التوليدي في ثلاثة أسطر.",
    "ترجم هذه الجملة للإنجليزية: الطقس اليوم جميل جداً في دمشق.",
    "اكتب لي دالة بايثون تتحقق إن كان الرقم أولياً.",
    "ما هو سعر الذهب اليوم؟",
    "أعطني نصائح لتعلم لغة برمجة جديدة بسرعة.",
    "ما الفرق بين HTTP و HTTPS؟",
    "اكتب لي رسالة اعتذار مهذبة لعميل تأخر طلبه.",
    "What's the difference between a list and a tuple in Python?",
    "Give me a short, friendly greeting in English.",
    "How do I reverse a string in JavaScript?",
    "اشرح ببساطة ماهو الـ API.",
    "قدّم لي خطة يومية لتعلم الإنجليزية خلال شهر.",
]


def _generate_groq_teacher_examples() -> list[dict]:
    try:
        groq_api_key = UserSecretsClient().get_secret("GROQ_API_KEY")
    except Exception:
        print("لا يوجد GROQ_API_KEY — سيُتخطى تقطير Groq (اختياري).")
        return []
    try:
        from groq import Groq
        client = Groq(api_key=groq_api_key)
    except Exception as e:
        print("تعذر تهيئة عميل Groq -", e)
        return []

    examples = []
    for prompt in _GROQ_SEED_PROMPTS:
        try:
            completion = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {"role": "system", "content": _NOVA_IDENTITY_PROMPT},
                    {"role": "user", "content": prompt},
                ],
            )
            reply = completion.choices[0].message.content
            if reply:
                examples.append({"question": prompt, "answer": reply.strip()})
        except Exception as e:
            print("تعذر توليد مثال لـ:", prompt, "-", e)
    print(f"Groq: تم توليد {len(examples)} مثال تدريب.")
    return examples


groq_teacher_examples = _generate_groq_teacher_examples()

In [ ]:
# الخلية 5ب — تقطير معرفي عبر Gemini كـ"معلّم" مجاني رقم 2 (وقت التدريب فقط)
#
# مصدر معرفة إضافي مستقل، وقت التدريب فقط. اختياري تماماً: بدون
# GEMINI_API_KEY في أسرار Kaggle، تُتخطى بأمان.
from kaggle_secrets import UserSecretsClient

_GEMINI_SEED_PROMPTS = [
    "ما هي أهم فوائد ممارسة الرياضة بانتظام؟",
    "اشرح لي مفهوم التضخم الاقتصادي ببساطة.",
    "أعطني فكرة مشروع صغير مربح يمكن البدء به بميزانية محدودة.",
    "ما الفرق بين الذكاء الاصطناعي والتعلم الآلي والتعلم العميق؟",
    "اكتب فقرة قصيرة تشجيعية لشخص بدأ للتو تعلم البرمجة.",
    "كيف أكتب سيرة ذاتية (CV) قوية لأول وظيفة؟",
    "What are three tips for staying productive while working from home?",
    "اشرح نظرية النسبية لأينشتاين بأبسط شكل ممكن لغير المختصين.",
    "ما هي أهم النقاط التي يجب مراعاتها عند تصميم واجهة مستخدم سهلة الاستخدام؟",
    "قدّم لي مقارنة موجزة بين قواعد البيانات العلائقية وغير العلائقية.",
]


def _generate_gemini_teacher_examples() -> list[dict]:
    try:
        gemini_api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception:
        print("لا يوجد GEMINI_API_KEY — سيُتخطى تقطير Gemini (اختياري).")
        return []
    try:
        import google.generativeai as genai
        genai.configure(api_key=gemini_api_key)
        # Gemini free-tier model names get retired over time (gemini-2.0-flash
        # then gemini-2.5-flash both were) — check
        # https://ai.google.dev/gemini-api/docs/models if this 404s.
        model = genai.GenerativeModel("gemini-3.6-flash", system_instruction=_NOVA_IDENTITY_PROMPT)
    except Exception as e:
        print("تعذر تهيئة عميل Gemini -", e)
        return []

    examples = []
    for prompt in _GEMINI_SEED_PROMPTS:
        try:
            response = model.generate_content(prompt)
            if response.text:
                examples.append({"question": prompt, "answer": response.text.strip()})
        except Exception as e:
            print("تعذر توليد مثال لـ:", prompt, "-", e)
    print(f"Gemini: تم توليد {len(examples)} مثال تدريب.")
    return examples


gemini_teacher_examples = _generate_gemini_teacher_examples()

In [ ]:
# الخلية 5ج — بيانات رؤية حقيقية من مجموعة بيانات عامة مفتوحة الترخيص
#
# هذا هو ما يُكسب النموذج فهم صور حقيقياً (وليس فقط نصاً يصف صوراً) —
# NovaUsageLog لا يخزّن صور المستخدمين إطلاقاً (سياسة خصوصية حالية:
# الصور تُرسل لـGemini للتحليل الفوري ثم تُحذف، لا تُحفظ)، فلا يوجد
# بديل عن مجموعة بيانات عامة لتوفير أمثلة صورة+سؤال+إجابة حقيقية.
#
# HuggingFaceM4/the_cauldron: تجميعة أكاديمية مجانية الترخيص لعدة
# مجموعات صور+أسئلة+إجابات (VQA عام). نأخذ عيّنة صغيرة فقط (500) لتناسب
# وقت/ذاكرة Kaggle الأسبوعية المجانية. اسم المجموعة/الإعداد الفرعي قد
# يتغيّر بمرور الوقت — تحقق من huggingface.co/datasets/HuggingFaceM4/the_cauldron
# إن فشل هذا التحميل مستقبلاً واستبدله بمجموعة عامة مشابهة.
try:
    from datasets import load_dataset

    _vision_ds = load_dataset("HuggingFaceM4/the_cauldron", "vqav2", split="train", streaming=True)
    vision_examples = []
    for _row in _vision_ds:
        if len(vision_examples) >= 500:
            break
        _texts = _row.get("texts") or []
        if not _texts or not _row.get("images"):
            continue
        _qa = _texts[0]
        vision_examples.append({
            "image": _row["images"][0],
            "question": _qa.get("user", "").strip(),
            "answer": _qa.get("assistant", "").strip(),
        })
    vision_examples = [e for e in vision_examples if e["question"] and e["answer"]]
    print(f"عدد أمثلة الرؤية الحقيقية المحمَّلة: {len(vision_examples)}")
except Exception as e:
    print("تعذر تحميل مجموعة بيانات الرؤية العامة -", e, "— سيُتخطى تدريب الرؤية هذه المرة (النص يستمر بشكل مستقل).")
    vision_examples = []

In [ ]:
# الخلية 5د — سيناريوهات وسائط إضافية كأمثلة نصية (تكميلية، وقت التدريب فقط)
#
# إضافة لأمثلة الرؤية الحقيقية أعلاه: سيناريوهات نصية إضافية (نص
# مفرَّغ من رسالة صوتية + سؤال) تُحسّن أسلوب الاستجابة في سياق الوسائط،
# خصوصاً الصوت الذي لا تغطيه مجموعة بيانات الرؤية أعلاه إطلاقاً.
from kaggle_secrets import UserSecretsClient

_MEDIA_SCENARIOS = [
    "نص مفرَّغ من رسالة صوتية أرسلها المستخدم: \"مرحبا نوفا، بدي مساعدة بكتابة رسالة اعتذار لصاحب عمل عن التأخير بتسليم مشروع\"",
    "نص مفرَّغ من رسالة صوتية أرسلها المستخدم: \"شو الفرق بين الذكاء الاصطناعي والتعلم الآلي بشكل مختصر؟\"",
    "نص مفرَّغ من رسالة صوتية أرسلها المستخدم: \"اكتب لي دالة بايثون بسيطة تحسب مجموع أرقام قائمة\"",
]


def _generate_media_scenario_examples() -> list[dict]:
    try:
        gemini_api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception:
        print("لا يوجد GEMINI_API_KEY — سيُتخطى تقطير سيناريوهات الوسائط (اختياري).")
        return []
    try:
        import google.generativeai as genai
        genai.configure(api_key=gemini_api_key)
        media_prompt = (
            _NOVA_IDENTITY_PROMPT
            + " ستصلك نصوص مفرَّغة من رسائل صوتية — تعامل معها كأنها المعلومة التي وصلتك فعلاً، وأجب مباشرة."
        )
        # Same model-retirement caveat as the Gemini teacher cell above.
        model = genai.GenerativeModel("gemini-3.6-flash", system_instruction=media_prompt)
    except Exception as e:
        print("تعذر تهيئة عميل Gemini لسيناريوهات الوسائط -", e)
        return []

    examples = []
    for scenario in _MEDIA_SCENARIOS:
        try:
            response = model.generate_content(scenario)
            if response.text:
                examples.append({"question": scenario, "answer": response.text.strip()})
        except Exception as e:
            print("تعذر توليد مثال لسيناريو وسائط -", e)
    print(f"سيناريوهات الوسائط: تم توليد {len(examples)} مثال تدريب.")
    return examples


media_teacher_examples = _generate_media_scenario_examples()

In [ ]:
# الخلية 5هـ — الدمج النهائي: كل مصادر البيانات معاً
text_training_data = real_text_examples + groq_teacher_examples + gemini_teacher_examples + media_teacher_examples
HAVE_TEXT_DATA = len(text_training_data) >= 5
HAVE_VISION_DATA = len(vision_examples) >= 5
HAVE_TRAINING_DATA = HAVE_TEXT_DATA or HAVE_VISION_DATA
print(f"أمثلة نصية: {len(text_training_data)} — أمثلة رؤية: {len(vision_examples)}")
if not HAVE_TRAINING_DATA:
    print("لا توجد بيانات كافية من أي نوع بعد — سيتم تخطي التدريب هذه المرة.")

In [ ]:
# الخلية 6 — تحميل Qwen2.5-VL عبر Unsloth للتدريب السريع (4-bit)
#
# ملاحظة أمانة: FastVisionModel هو واجهة Unsloth الأحدث لنماذج
# الرؤية-اللغة (Qwen2-VL/Qwen2.5-VL، Llava، Pixtral، وغيرها) — إن تغيّر
# اسم الدالة أو معاملاتها في إصدار Unsloth مستقبلاً، راجع
# https://docs.unsloth.ai لأحدث توقيع الدالة قبل افتراض أن هذا لا يزال
# صحيحاً حرفياً (نفس تحذير GROQ_MODEL/GEMINI_MODEL المتكرر في هذا
# المشروع: كتالوجات ومكتبات مزوّدي الذكاء الاصطناعي المجانيين تتغيّر
# بمرور الوقت).
if HAVE_TRAINING_DATA:
    from unsloth import FastVisionModel

    model, tokenizer = FastVisionModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        load_in_4bit=True,
        max_seq_length=2048,
    )
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=HAVE_VISION_DATA,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
    )
    print("تم تحميل", BASE_MODEL_ID, "بنجاح للتدريب.")
else:
    print("تخطي تحميل النموذج — لا توجد بيانات كافية بعد.")

In [ ]:
# الخلية 7 — تجهيز بيانات موحّدة (نص + صورة) والتدريب الفعلي
if HAVE_TRAINING_DATA:
    conversations = []

    for ex in text_training_data:
        conversations.append({
            "messages": [
                {"role": "user", "content": [{"type": "text", "text": ex["question"]}]},
                {"role": "assistant", "content": [{"type": "text", "text": ex["answer"]}]},
            ]
        })

    for ex in vision_examples:
        conversations.append({
            "messages": [
                {"role": "user", "content": [
                    {"type": "image", "image": ex["image"]},
                    {"type": "text", "text": ex["question"]},
                ]},
                {"role": "assistant", "content": [{"type": "text", "text": ex["answer"]}]},
            ]
        })

    print(f"إجمالي محادثات التدريب الموحّدة (نص + رؤية): {len(conversations)}")

    from unsloth.trainer import UnslothVisionDataCollator
    from trl import SFTTrainer, SFTConfig

    FastVisionModel.for_training(model)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=conversations,
        args=SFTConfig(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            num_train_epochs=1,
            learning_rate=2e-4,
            output_dir="./nova-vl-lora-out",
            logging_steps=1,
            remove_unused_columns=False,
            dataset_text_field="",
            max_seq_length=2048,
        ),
    )
    trainer.train()
else:
    print("تخطي التدريب — لا توجد بيانات كافية بعد.")

In [ ]:
# الخلية 8 — حفظ محلي + رفع النموذج المدرَّب إلى مستودعنا الخاص على Hugging Face
#
# فشل حقيقي تكرر مرتين في الخلايا التالية رغم محاولات تقليل الحجم:
# "OSError: No space left on device". السبب الجذري الحقيقي (بحثاً
# مباشراً، وليس افتراضاً): /kaggle/working له سقف صارم ~20GB فقط لأن
# Kaggle يعامله كـ"مخرجات" دائمة للدفتر، بينما /tmp على نفس الحاوية
# يوفر ~60GB (مساحة عمل مؤقتة لا تُحتسب ضمن حصة المخرجات). لا شيء من
# الملفات الكبيرة هنا (النموذج المدموج، ملفات GGUF) يحتاج أن يكون
# "مخرجاً" دائماً للدفتر أصلاً — كلها تُرفع مباشرة لـHugging Face
# وModelScope ضمن نفس التشغيل، فنقلها جميعاً إلى /tmp (هنا وفي الخلايا
# 9، 10، 11) يحل مشكلة القرص جذرياً بدل تقليصها هامشياً فقط.
LOCAL_MERGED_DIR = "/tmp/nova-vl-merged"

if HAVE_TRAINING_DATA:
    FastVisionModel.for_inference(model)
    model.save_pretrained_merged(LOCAL_MERGED_DIR, tokenizer, save_method="merged_16bit")

    from huggingface_hub import create_repo, upload_folder

    create_repo(REPO_ID, exist_ok=True)
    upload_folder(repo_id=REPO_ID, folder_path=LOCAL_MERGED_DIR)
    print(f"تم رفع نموذجنا الموحّد (نص + رؤية) إلى: https://huggingface.co/{REPO_ID}")
else:
    print("تخطي الرفع — لم يُجرَ تدريب هذه المرة.")

---
## تحويل GGUF + mmproj، ورفع صيغة التشغيل الفعلية إلى ModelScope

**لماذا هذه الخطوة ضرورية؟** خادم الاستضافة الفعلي (ModelScope
Studio) يشغّل `llama-cpp-python` على معالج عادي (CPU) فقط بلا بطاقة
رسومية — وهذا يحتاج صيغة GGUF، وليس صيغة `safetensors` العادية
المرفوعة أعلاه لـHugging Face (تلك نسخة الملكية/الأرشفة، وليست صيغة
تشغيل CPU).

**تحقّقتُ (بحثاً حقيقياً، وليس افتراضاً) أن تصدير GGUF+mmproj الخاص
بنماذج الرؤية داخل Unsloth نفسها (`save_pretrained_gguf` لِـ
`FastVisionModel`) لا يزال تجريبياً وغير مكتمل رسمياً (PR رقم 1904 في
مستودع Unsloth، لا يزال Draft/مغلقاً، ومُبلَّغ عنه أنه يُنتج مخرجات
تالفة "GGGGGGG..." عند دمج ملف mmproj المُستخرَج مع نموذج مدرَّب) —
لذلك **لا نعتمد عليه**. البديل الموثوق والمُستخدَم مجتمعياً هو أداة
`llama.cpp` الرسمية نفسها (`convert_hf_to_gguf.py`)، التي تدعم
Qwen2.5-VL فعلياً عبر تشغيلها **مرتين**: مرة عادية لملف النموذج
الرئيسي، ومرة بخيار `--mmproj` لاستخراج "مُرمِّز الرؤية" في ملف منفصل
— وهذا ما تفعله الخلايا التالية بالضبط.

**لماذا نحتاج نسخة `llama-cpp-python` من JamePeng على خادم
ModelScope (وليس النسخة الرسمية من PyPI)؟** تحقّقتُ أن نسخة PyPI
الرسمية (`abetlen/llama-cpp-python`) لا تتضمن معالِج محادثة (Chat
Handler) لعائلة Qwen2.5-VL إطلاقاً حتى تاريخ 2026-09-07 — نسخة
JamePeng (`github.com/JamePeng/llama-cpp-python`) هي التي تضيف
`Qwen25VLChatHandler` فعلياً، وتُبنى من المصدر عبر معالج عادي بلا
حاجة لبطاقة رسومية. تحديث `app.py`/`requirements.txt` الخاصين بخادم
ModelScope نفسه (ملفان خارج مستودع Git هذا، يُرفعان يدوياً من الهاتف)
سيُرسَلان جاهزين للنسخ بعد نجاح هذا الدفتر.

**مرة واحدة فقط قبل أول تشغيل لهذا الجزء:** أضف Kaggle Secret باسم
`MODELSCOPE_TOKEN` (توكن بصيغة "ms-xxxxx" من modelscope.cn -> إعدادات
الحساب -> Access Tokens) وآخر باسم `MODELSCOPE_USERNAME` (اسم حسابك
هناك، مثل "novaai2026").

In [ ]:
# الخلية 9 — بناء أداة llama.cpp الرسمية للتحويل (مرة واحدة في كل تشغيل)
#
# في /tmp وليس /kaggle/working (انظر ملاحظة الخلية 8 عن سقف القرص).
# استنساخ ضحل (depth=1) فقط لتوفير الوقت — لا نحتاج تاريخ المستودع
# الكامل، فقط أحدث نسخة من سكربتات التحويل والبناء.
if HAVE_TRAINING_DATA:
    import os

    if not os.path.isdir("/tmp/llama.cpp"):
        !git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
    !pip install -q -r /tmp/llama.cpp/requirements.txt
    # llama-quantize (لضغط q8_0 -> q4_k_m لاحقاً) يحتاج بناء C++ فعلي
    # عبر cmake — لا نحتاج بطاقة رسومية لهذا، البناء عادي على المعالج.
    !cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF
    !cmake --build /tmp/llama.cpp/build --target llama-quantize --config Release -j
    print("تم تجهيز llama.cpp للتحويل والضغط.")
else:
    print("تخطي — لم يُجرَ تدريب هذه المرة.")

In [ ]:
# الخلية 10 — التحويل الفعلي: نموذج رئيسي GGUF (مضغوط q4_k_m) + ملف mmproj منفصل للرؤية
#
# خطوتان منفصلتان حرفياً كما توثّق llama.cpp نفسها لهذه العائلة من
# النماذج: تحويل عادي بدون --mmproj ينتج النموذج اللغوي فقط؛ تحويل
# بخيار --mmproj ينتج "مُرمِّز الرؤية" فقط. الاثنان مطلوبان معاً على
# خادم التشغيل (Qwen25VLChatHandler يحمّلهما كملفين منفصلين).
#
# فشل حقيقي تكرر هنا مرتين رغم محاولتي الأولى تقليل الحجم (f16 -> q8_0
# كصيغة وسيطة): "OSError: No space left on device" — تبيّن بالبحث أن
# السبب الجذري ليس حجم الملفات بل أن /kaggle/working نفسه له سقف صارم
# ~20GB (Kaggle يعامله كمخرجات دائمة)، بينما /tmp على نفس الحاوية
# يوفر ~60GB. الإصلاح الحقيقي: كل الملفات هنا في /tmp الآن (انظر ملاحظة
# الخلية 8) — لا حاجة أصلاً لتصغير الصيغ الوسيطة إن كانت المساحة
# كافية، لكن أبقينا q8_0 بدل f16 كطبقة أمان إضافية رخيصة.
#
# ملاحظة أمانة أخرى اكتُشفت حياً: أوامر الشل (!...) في Jupyter لا
# تُطلق خطأ بايثون حتى لو فشلت فعلياً — هذا بالضبط ما جعل فشل القرص
# يمر دون أن يوقف الخلية في المرتين السابقتين، ولم يظهر إلا لاحقاً
# كخطأ غامض في خلية تالية. أضفنا تحققاً صريحاً بالأسفل يوقف التنفيذ
# بخطأ حقيقي وواضح إن لم يُنتَج الملف فعلياً، بدل الاستمرار بصمت.
if HAVE_TRAINING_DATA:
    import os
    import shutil

    GGUF_Q8_PATH = "/tmp/nova-vision-7b-q8_0.gguf"
    GGUF_Q4_PATH = "/tmp/nova-vision-7b-q4_k_m.gguf"
    MMPROJ_PATH = "/tmp/mmproj-nova-vision-7b-f16.gguf"

    shutil.rmtree("/root/.cache/huggingface", ignore_errors=True)

    !python /tmp/llama.cpp/convert_hf_to_gguf.py {LOCAL_MERGED_DIR} --outfile {GGUF_Q8_PATH} --outtype q8_0
    !python /tmp/llama.cpp/convert_hf_to_gguf.py {LOCAL_MERGED_DIR} --mmproj --outfile {MMPROJ_PATH} --outtype f16

    # كلا التحويلين انتهى من قراءة المجلد المدموج — نحذفه الآن لتوفير
    # مساحة قبل خطوة الضغط (Quantize) التالية.
    shutil.rmtree(LOCAL_MERGED_DIR, ignore_errors=True)

    !/tmp/llama.cpp/build/bin/llama-quantize {GGUF_Q8_PATH} {GGUF_Q4_PATH} q4_k_m
    if os.path.exists(GGUF_Q8_PATH):
        os.remove(GGUF_Q8_PATH)

    for _path in (GGUF_Q4_PATH, MMPROJ_PATH):
        if not os.path.exists(_path):
            raise RuntimeError(
                f"فشل التحويل فعلياً — الملف {_path} لم يُنشأ. راجع مخرجات الخلية أعلاه لمعرفة الخطأ الحقيقي (غالباً في سطور convert_hf_to_gguf.py أو llama-quantize)."
            )

    print("النموذج الرئيسي (مضغوط):", GGUF_Q4_PATH)
    print("ملف الرؤية (mmproj):", MMPROJ_PATH)
else:
    print("تخطي — لم يُجرَ تدريب هذه المرة.")

In [ ]:
# الخلية 11 — رفع ملفَي GGUF إلى مستودعنا الخاص على ModelScope
#
# هذا هو المستودع الذي يقرأ منه app.py الفعلي على خادم ModelScope —
# منفصل تماماً عن مستودع Hugging Face أعلاه (ذاك للأرشفة/الملكية،
# هذا لصيغة التشغيل الفعلية). في /tmp وليس /kaggle/working (انظر
# ملاحظة الخلية 8 عن سقف القرص).
if HAVE_TRAINING_DATA:
    import json
    import os
    import shutil

    !pip install -q modelscope

    from kaggle_secrets import UserSecretsClient
    from modelscope.hub.api import HubApi
    from modelscope.hub.constants import ModelVisibility

    _ms_secrets = UserSecretsClient()
    MODELSCOPE_TOKEN = _ms_secrets.get_secret("MODELSCOPE_TOKEN")
    MODELSCOPE_USERNAME = _ms_secrets.get_secret("MODELSCOPE_USERNAME")
    MODELSCOPE_MODEL_REPO = f"{MODELSCOPE_USERNAME}/nova-vision-7b-gguf"

    # push_model يتطلب configuration.json داخل مجلد الرفع — أدنى شكل
    # مقبول لمستودع نموذج عام على ModelScope.
    _upload_dir = "/tmp/nova-vision-7b-gguf-upload"
    os.makedirs(_upload_dir, exist_ok=True)
    with open(os.path.join(_upload_dir, "configuration.json"), "w") as f:
        json.dump({"framework": "pytorch", "task": "text-generation"}, f)
    shutil.copy(GGUF_Q4_PATH, _upload_dir)
    shutil.copy(MMPROJ_PATH, _upload_dir)

    _api = HubApi()
    _api.login(MODELSCOPE_TOKEN)
    _api.push_model(
        model_id=MODELSCOPE_MODEL_REPO,
        model_dir=_upload_dir,
        visibility=ModelVisibility.PUBLIC,
        license="apache-2.0",
    )
    print(f"تم رفع صيغة التشغيل (GGUF + mmproj) إلى: https://modelscope.cn/models/{MODELSCOPE_MODEL_REPO}")
    print("ضع هذا في MODEL_ID داخل app.py على خادم ModelScope:", MODELSCOPE_MODEL_REPO)
else:
    print("تخطي — لم يُجرَ تدريب هذه المرة.")